In [2]:
from load_ud_dataset import load_ud_dataset, UD_GERMAN_SPLITS

In [3]:
UD_GERMAN_GSD_SPLITS = {
    'train': 'de_gsd-ud-train.conllu',
    'dev': 'de_gsd-ud-dev.conllu',
    'test': 'de_gsd-ud-test.conllu',
}
ud_path = "../data/UD_German-GSD"

In [4]:
ud = load_ud_dataset(ud_path, splits_filemap=UD_GERMAN_SPLITS[0])

/opt/anaconda3/envs/morph/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded train: 13813 sentences
Loaded dev: 799 sentences
Loaded test: 977 sentences


In [5]:
ud

DatasetDict({
    train: Dataset({
        features: ['tokens', 'sent_id', 'text'],
        num_rows: 13813
    })
    dev: Dataset({
        features: ['tokens', 'sent_id', 'text'],
        num_rows: 799
    })
    test: Dataset({
        features: ['tokens', 'sent_id', 'text'],
        num_rows: 977
    })
})

In [6]:
ud['test'][0]

{'tokens': [{'deprel': 'det',
   'deps': '_',
   'feats': {'Case': 'Nom',
    'Definite': 'Def',
    'Degree': None,
    'ExtPos': None,
    'Foreign': None,
    'Gender': 'Masc',
    'Gender[psor]': None,
    'Mood': None,
    'NumType': None,
    'Number': 'Sing',
    'Number[psor]': None,
    'Person': None,
    'Polarity': None,
    'Polite': None,
    'Poss': None,
    'PronType': 'Art',
    'Reflex': None,
    'Tense': None,
    'Typo': None,
    'VerbForm': None,
    'Voice': None},
   'feats_str': 'Case=Nom|Definite=Def|Gender=Masc|Number=Sing|PronType=Art',
   'form': 'Der',
   'head': '2',
   'id': '1',
   'lemma': 'der',
   'misc': '_',
   'upos': 'DET',
   'xpos': 'ART'},
  {'deprel': 'nsubj',
   'deps': '_',
   'feats': {'Case': 'Nom',
    'Definite': None,
    'Degree': None,
    'ExtPos': None,
    'Foreign': None,
    'Gender': 'Masc',
    'Gender[psor]': None,
    'Mood': None,
    'NumType': None,
    'Number': 'Sing',
    'Number[psor]': None,
    'Person': None,
   

In [12]:
from conllu import parse_incr
import pandas as pd

########################################################################
# CONFIG
########################################################################

DAT_POS = {"NOUN", "PROPN", "PRON"}

########################################################################
# HELPERS
########################################################################

def has_dative(feats):
    if feats is None:
        return False
    return feats.get("Case") == "Dat"


def get_token_by_id(sent, idx):
    for tok in sent:
        if tok["id"] == idx:
            return tok
    return None


def get_children(sent, head_id):
    return [tok for tok in sent if tok["head"] == head_id]


def collect_span(sent, head_id):
    """
    Collect a simple NP span around the dative head (no preposition).
    Prepositions are stored separately via get_preposition().
    """

    allowed = {
        "det",
        "amod",
        "nummod",
        "compound",
        "fixed",
        "flat",
    }

    span_tokens = []

    for tok in sent:
        if tok["id"] == head_id:
            span_tokens.append(tok)

        elif tok["head"] == head_id and tok["deprel"] in allowed:
            span_tokens.append(tok)

    span_tokens = sorted(span_tokens, key=lambda x: x["id"])

    return span_tokens


def get_preposition(sent, head_id):
    for tok in sent:
        if tok["head"] == head_id and tok["deprel"] == "case":
            return tok["lemma"]
    return None


def classify_dative(token, prep):

    rel = token["deprel"]

    if prep is not None:
        return "prepositional_dative"

    if rel == "iobj":
        return "indirect_object"

    if rel in {"obj", "obl:arg"}:
        return "core_dative_argument"

    if rel == "obl":
        return "oblique_dative"

    if rel == "nmod":
        return "nominal_dative_modifier"

    return "other_dative"

In [ ]:

########################################################################
# MAIN
########################################################################

rows = []

with open("../data/UD_German-GSD/de_gsd-ud-train.conllu", "r", encoding="utf-8") as f:

    for sent in parse_incr(f):

        sent_id = sent.metadata.get("sent_id", "")
        text = sent.metadata.get("text", "")

        for tok in sent:

            if tok["upos"] not in DAT_POS:
                continue

            if not has_dative(tok["feats"]):
                continue

            span = collect_span(sent, tok["id"])

            span_text = " ".join(t["form"] for t in span)

            prep = get_preposition(sent, tok["id"])

            gov = get_token_by_id(sent, tok["head"])

            if gov is None:
                gov_form = "ROOT"
                gov_upos = "ROOT"
            else:
                gov_form = gov["form"]
                gov_upos = gov["upos"]

            subtype = classify_dative(tok, prep)

            rows.append({
                "sent_id": sent_id,
                "sentence": text,

                "span": span_text,
                "head_form": tok["form"],
                "head_lemma": tok["lemma"],
                "head_upos": tok["upos"],

                "preposition": prep,
                "dative_type": subtype,

                "relation": tok["deprel"],

                "governor": gov_form,
                "governor_upos": gov_upos,
            })

df = pd.DataFrame(rows)

print(df.head())

df.to_csv("dative_spans/german_datives_train.csv", index=False)

    sent_id                                           sentence  \
0  train-s1  Sehr gute Beratung, schnelle Behebung der Prob...   
1  train-s2          Die Kosten sind definitiv auch im Rahmen.   
2  train-s4  Ich bin seit längerer Zeit zur Behandlung vers...   
3  train-s4  Ich bin seit längerer Zeit zur Behandlung vers...   
4  train-s4  Ich bin seit längerer Zeit zur Behandlung vers...   

                      span            head_form           head_lemma  \
0                      mir                  mir                  ich   
1               dem Rahmen               Rahmen               Rahmen   
2            längerer Zeit                 Zeit                 Zeit   
3           der Behandlung           Behandlung           Behandlung   
4  der Physiotherapieraxis  Physiotherapieraxis  Physiotherapieraxis   

  head_upos preposition           dative_type relation governor governor_upos  
0      PRON        None  core_dative_argument  obl:arg   stelle          VERB  
1      NOU

In [10]:
set(df.dative_type)

{'core_dative_argument',
 'nominal_dative_modifier',
 'oblique_dative',
 'other_dative',
 'prepositional_dative'}

In [ ]:

########################################################################
# MAIN
########################################################################

splits = {
    'train': '../data/UD_German-GSD/de_gsd-ud-train.conllu',
    'dev': '../data/UD_German-GSD/de_gsd-ud-dev.conllu',
    'test': '../data/UD_German-GSD/de_gsd-ud-test.conllu',
}

all_dfs = []

for split in splits:
    rows = []

    with open(splits[split], "r", encoding="utf-8") as f:

        for sent in parse_incr(f):

            sent_id = sent.metadata.get("sent_id", "")
            text = sent.metadata.get("text", "")

            for tok in sent:

                if tok["upos"] not in DAT_POS:
                    continue

                if not has_dative(tok["feats"]):
                    continue

                span = collect_span(sent, tok["id"])

                span_text = " ".join(t["form"] for t in span)

                prep = get_preposition(sent, tok["id"])

                gov = get_token_by_id(sent, tok["head"])

                if gov is None:
                    gov_form = "ROOT"
                    gov_upos = "ROOT"
                else:
                    gov_form = gov["form"]
                    gov_upos = gov["upos"]

                subtype = classify_dative(tok, prep)

                rows.append({
                    "sent_id": sent_id,
                    "sentence": text,

                    "span": span_text,
                    "head_form": tok["form"],
                    "head_lemma": tok["lemma"],
                    "head_upos": tok["upos"],

                    "preposition": prep,
                    "dative_type": subtype,

                    "relation": tok["deprel"],

                    "governor": gov_form,
                    "governor_upos": gov_upos,
                    "split": split,  # add split information for provenance
                })

    df = pd.DataFrame(rows)
    all_dfs.append(df)

# Combine into unified DataFrame
unified_df = pd.concat(all_dfs, ignore_index=True)

print(unified_df.head())

unified_df.to_csv("dative_spans/german_datives_all.csv", index=False)

    sent_id                                           sentence  \
0  train-s1  Sehr gute Beratung, schnelle Behebung der Prob...   
1  train-s2          Die Kosten sind definitiv auch im Rahmen.   
2  train-s4  Ich bin seit längerer Zeit zur Behandlung vers...   
3  train-s4  Ich bin seit längerer Zeit zur Behandlung vers...   
4  train-s4  Ich bin seit längerer Zeit zur Behandlung vers...   

                      span            head_form           head_lemma  \
0                      mir                  mir                  ich   
1               dem Rahmen               Rahmen               Rahmen   
2            längerer Zeit                 Zeit                 Zeit   
3           der Behandlung           Behandlung           Behandlung   
4  der Physiotherapieraxis  Physiotherapieraxis  Physiotherapieraxis   

  head_upos preposition           dative_type relation governor governor_upos  \
0      PRON        None  core_dative_argument  obl:arg   stelle          VERB   
1      N

In [25]:
set(unified_df.preposition)

{"'",
 "'s",
 'Nach',
 None,
 'ab',
 'abseits',
 'als',
 'an',
 'anhand',
 'anläßlich',
 'anstatt',
 'anstelle',
 'auf',
 'aufgrund',
 'aus',
 'außer',
 'außerhalb',
 'bei',
 'binnen',
 'bis',
 'dank',
 'durch',
 'ei',
 'einschließlich',
 'entgegen',
 'entlang',
 'entsprechend',
 'für',
 'gegen',
 'gegenüber',
 'gemäß',
 'hinsichtlich',
 'hinter',
 'in',
 'infolge',
 'inklusive',
 'inmitten',
 'innerhalb',
 'je',
 'laut',
 'längs',
 'mit',
 'mitsamt',
 'mittels',
 'nach',
 'nahe',
 'namens',
 'neben',
 'nebst',
 'nordwestlich',
 'nördlich',
 'ob',
 'oberhalb',
 'ohne',
 'per',
 'pro',
 'samt',
 'seit',
 'seitens',
 'statt',
 'südlich',
 'trotz',
 'u.a.',
 'um',
 'unter',
 'unterhalb',
 'unweit',
 'via',
 'voll',
 'von',
 'vor',
 'vorbei',
 'vum',
 'wegen',
 'westlich',
 'wie',
 'während',
 'z.',
 'zu',
 'zufolge',
 'zugunsten',
 'zuzüglich',
 'zwischen',
 'über'}

In [21]:
lookup_dict = pd.read_csv("german_ud_cases_dictionary.csv")
lookup_dict.head()

,Lemma,Case,Number,Gender,Upos,Forms
0,A,Dat,Sing,Neut,PROPN,A
1,A,Dat,Sing,Fem,PROPN,A
2,A,Dat,Sing,Fem,NOUN,A
3,A,Acc,Sing,NaN,PROPN,A
4,A,Dat,Sing,Masc,PROPN,A


In [22]:
set(lookup_dict.Upos)

{'ADJ', 'ADP', 'ADV', 'DET', 'NOUN', 'NUM', 'PRON', 'PROPN', 'X'}

In [23]:
# how many rows has no Gender?
lookup_dict[lookup_dict.Gender.isna()]


,Lemma,Case,Number,Gender,Upos,Forms
3,A,Acc,Sing,NaN,PROPN,A
8,A.V.G.,Dat,Sing,NaN,PROPN,A.V.G.
14,A36,Acc,Sing,NaN,PROPN,A36
20,ABB,Dat,Sing,NaN,PROPN,ABB
23,ABC,Dat,Sing,NaN,PROPN,ABC
...,...,...,...,...,...,...
118469,üblich,Dat,Plur,NaN,ADJ,üblichen
118475,üblich,Acc,Plur,NaN,ADJ,übliche
118479,übrig,Dat,Plur,NaN,ADJ,übrigen
118487,übrig,Dat,Sing,NaN,ADJ,übrigen


In [24]:
# how many rows has no Number?
lookup_dict[lookup_dict.Number.isna()]


,Lemma,Case,Number,Gender,Upos,Forms
47,ACT,Acc,NaN,NaN,PROPN,ACT
59,ADSL,Acc,NaN,NaN,NOUN,ADSL
110,AFR,Dat,NaN,NaN,NOUN,AFR
117,AGBs,Dat,NaN,NaN,NOUN,AGBs
154,AIDS,Dat,NaN,NaN,NOUN,AIDS
...,...,...,...,...,...,...
118195,über,Dat,NaN,NaN,ADP,über
118373,überschreiben,Dat,NaN,NaN,ADJ,überschrieben
118433,überwiegend,Dat,NaN,NaN,ADJ,überwiegend
118464,üblich,Dat,NaN,NaN,ADJ,üblich
